# Шаг 4. Э2 — межъязыковой перенос (английский → русский)

Вопрос: можно ли обучить детектор на английских текстах и применить к русским, и сколько русских примеров нужно, чтобы он заработал.

Что делает ноутбук (≈40–50 минут, почти всё время — ячейка 6):
1. **Детекторы на признаках** (стилометрический, статистический, гибридный): обучение только на английском и проверка на русском; затем дообучение на 100, 500, 1000 русских примерах. Занимает пару минут, языковые модели не запускаются (берётся кэш шага 2).
2. **Многоязычный энкодер XLM-RoBERTa-base** — четвёртый метод из раздела 2.3. Дообучается на английских текстах, проверяется на русских без русских примеров, затем доучивается на 100, 500, 1000 русских примерах. Для сравнения такой же энкодер обучается только на русских примерах.
3. Каждый опыт с малым числом примеров повторяется 3 раза на разных случайных наборах примеров — результат на 100 примерах сильно зависит от того, какие именно примеры попались.

Русская тестовая выборка — **та же**, что в Э1 (1350 текстов), поэтому результаты можно сравнивать с таблицей 3.2 диплома.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**. Затем «Выполнить все».
Промежуточные результаты сохраняются на Диск после каждого опыта — если Colab отключится, сделанное не пропадёт.

**Прислать:** вывод ячеек 4–7, файл `results_step4.json` и рисунок `e2_curve.png` (оба в папке `ai-text-detector` на Диске).

## 1. Подготовка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, torch
WORK = '/content/drive/MyDrive/ai-text-detector'
os.makedirs('/content/code', exist_ok=True)
os.chdir('/content/code'); sys.path.insert(0, '/content/code')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
assert DEVICE == 'cuda', 'Нужна среда с GPU: Среда выполнения → Сменить среду выполнения → T4 GPU'
import transformers; print('transformers', transformers.__version__)

## 2. Код проекта

In [ ]:
%%writefile /content/code/stylometry.py
"""Стилометрические признаки (та же логика будет реализована в расширении на JavaScript)."""
import re, math, unicodedata
from collections import Counter

RU_FUNC = set("и в во не на я что он с со как а то все она так его но да ты к ко у же вы за бы по только ее её мне было вот от меня еще ещё нет о об из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между".split())
EN_FUNC = set("the of and to a in is that it for on as with was be by this are or from at an but not have has had which they you he she we his her their its can will would there been were if all so no one more also do than about into such these other only some may then".split())

CYR2LAT = {"А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H","О":"O","Р":"P","С":"C","Т":"T","Х":"X",
           "а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x"}
LAT2CYR = {v: k for k, v in CYR2LAT.items()}

def cyr_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('Ѐ' <= ch <= 'ӿ' for ch in letters) / len(letters)

def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('a' <= ch.lower() <= 'z' for ch in letters) / len(letters)

def detect_lang(text):
    """'ru' — преобладает кириллица, 'en' — почти только латиница, иначе 'other' (не поддерживается)."""
    if cyr_ratio(text) > 0.5: return 'ru'
    if latin_ratio(text) > 0.9: return 'en'
    return 'other'

def _convert(word, table):
    return ''.join(table.get(ch, ch) for ch in word)

def _mixed(word):
    return any('\u0400' <= ch <= '\u04FF' for ch in word) and any('a' <= ch.lower() <= 'z' for ch in word)

def _fix_word(word, doc_script):
    """Слово из смеси алфавитов приводим к основному алфавиту текста; если это невозможно
    (в слове есть буквы без двойников) — к алфавиту большинства букв слова (защита от гомоглифов)."""
    if not _mixed(word):
        # слово целиком из «чужого» алфавита, но только из букв-двойников (например, кириллическая «а» в английском тексте)
        foreign = CYR2LAT if doc_script == 'lat' else LAT2CYR
        if word and all(ch in foreign for ch in word):
            return _convert(word, foreign)
        return word
    first = _convert(word, LAT2CYR if doc_script == 'cyr' else CYR2LAT)
    if not _mixed(first):
        return first
    cyr = sum('\u0400' <= ch <= '\u04FF' for ch in word)
    lat = sum('a' <= ch.lower() <= 'z' for ch in word)
    return _convert(word, LAT2CYR if cyr > lat else CYR2LAT)

def normalize(text):
    """NFKC, удаление невидимых символов, нормализация пробелов, исправление гомоглифов."""
    t = unicodedata.normalize('NFKC', text)
    t = re.sub(r'[\u200b\u200c\u200d\u2060\ufeff]', '', t)
    t = re.sub(r'[ \t\u00a0]+', ' ', t)
    t = re.sub(r'\s*\n\s*', '\n', t).strip()
    doc_script = 'cyr' if cyr_ratio(t) > 0.5 else 'lat'
    return re.sub(r'\w+', lambda m: _fix_word(m.group(0), doc_script), t)

SENT_SPLIT = re.compile(r'(?<=[.!?…])\s+')
WORD = re.compile(r"[^\W\d_]+(?:[-'’][^\W\d_]+)*", re.UNICODE)

FEATURE_NAMES = ['mean_sent_len', 'cv_sent_len', 'mean_word_len', 'ttr', 'punct_ratio',
                 'func_ratio', 'rep_bigram_ratio', 'rep_trigram_ratio', 'comma_per_sent']

def stylometric_features(text, lang=None):
    lang = lang or detect_lang(text)
    sents = [s for s in SENT_SPLIT.split(text) if s.strip()]
    words = [w.lower() for w in WORD.findall(text)]
    n = max(len(words), 1)
    slens = [len(WORD.findall(s)) for s in sents] or [0]
    mu = sum(slens) / len(slens)
    sd = math.sqrt(sum((x - mu) ** 2 for x in slens) / len(slens))
    func = RU_FUNC if lang == 'ru' else EN_FUNC
    def rep(k):
        grams = [tuple(words[i:i+k]) for i in range(len(words) - k + 1)]
        if not grams: return 0.0
        c = Counter(grams); return sum(v for v in c.values() if v > 1) / len(grams)
    chars = max(len(text), 1)
    return {
        'mean_sent_len': mu,
        'cv_sent_len': sd / mu if mu else 0.0,
        'mean_word_len': sum(map(len, words)) / n,
        'ttr': len(set(words)) / n,
        'punct_ratio': sum(unicodedata.category(ch).startswith('P') for ch in text) / chars,
        'func_ratio': sum(w in func for w in words) / n,
        'rep_bigram_ratio': rep(2),
        'rep_trigram_ratio': rep(3),
        'comma_per_sent': text.count(',') / max(len(sents), 1),
    }

In [ ]:
%%writefile /content/code/metrics_utils.py
"""Метрики качества детектора (раздел 1.6)."""
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, precision_score, recall_score

def tpr_at_fpr(y, p, fpr_target=0.01):
    fpr, tpr, _ = roc_curve(y, p)
    ok = fpr <= fpr_target
    return float(tpr[ok].max()) if ok.any() else 0.0

def evaluate(y, p, threshold=0.5, n_boot=200, seed=0):
    y, p = np.asarray(y), np.asarray(p); yhat = (p >= threshold).astype(int)
    res = {'n': int(len(y)), 'F1': f1_score(y, yhat), 'Precision': precision_score(y, yhat, zero_division=0),
           'Recall': recall_score(y, yhat), 'FPR': float(((yhat == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)),
           'AUROC': roc_auc_score(y, p), 'TPR@1%FPR': tpr_at_fpr(y, p)}
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):                       # доверительный интервал AUROC (бутстреп)
        idx = rng.integers(0, len(y), len(y))
        if len(set(y[idx])) == 2: boots.append(roc_auc_score(y[idx], p[idx]))
    res['AUROC_95CI'] = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))) if boots else None
    return res

In [ ]:
%%writefile /content/code/baseline_stylometry.py
"""Базовый стилометрический детектор: признаки + логистическая регрессия, отдельно для каждого языка."""
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from stylometry import stylometric_features, normalize, FEATURE_NAMES
from metrics_utils import evaluate

def _features_with_format(text, lang):
    t = normalize(text)
    f = stylometric_features(t, lang)
    f['newline_ratio'] = t.count('\n') / max(len(t), 1)   # признак форматирования — только для эксперимента-абляции
    return f

def featurize(df, names=FEATURE_NAMES):
    return np.array([[f[k] for k in names] for f in
                     (_features_with_format(t, l) for t, l in zip(df.text, df.lang))])

def split(df, seed=42):
    """70/15/15 с разбиением по pair_id: человеческий текст и его сгенерированная пара не разделяются."""
    g1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    tr, rest = next(g1.split(df, groups=df.pair_id))
    rest_df = df.iloc[rest]
    g2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    va, te = next(g2.split(rest_df, groups=rest_df.pair_id))
    return df.iloc[tr], rest_df.iloc[va], rest_df.iloc[te]

def run(df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    d = df[df.lang == lang]
    d = pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])
    if d.label.nunique() < 2:
        return None, None, None
    tr, va, te = split(d, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'val': len(va), 'test': len(te)})
    coefs = pd.Series(model[-1].coef_[0], index=names).sort_values(key=abs, ascending=False)
    return res, coefs, model

def export_weights(models, path):
    """Сохраняет параметры нормировки и логистической регрессии в JSON для расширения (extension/model/weights.json)."""
    import json
    out = {'version': 'stylometry-v1', 'demo': False, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        out['langs'][lang] = {'features': FEATURE_NAMES, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0])}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=1)

def balanced_sample(d, cap_per_class, seed=42):
    return pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])

def run_cross(train_df, test_df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    """Обучение на одном корпусе и проверка на другом (перенос на незнакомые генераторы, серия Э3)."""
    tr = balanced_sample(train_df[train_df.lang == lang], cap_per_class, seed)
    te = balanced_sample(test_df[test_df.lang == lang], cap_per_class // 4, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'test': len(te)})
    return res

In [ ]:
%%writefile /content/code/lm_features.py
"""Вероятностные признаки по компактной авторегрессионной языковой модели (раздел 2.3, формулы 2.1–2.2).
Для каждого токена считаются: логарифм вероятности, энтропия предсказанного распределения, ранг
фактического токена и дисперсия логарифма вероятности (для признака условной кривизны, как в Fast-DetectGPT)."""
import numpy as np

LM_FEATURE_NAMES = ['mean_lp', 'std_lp', 'mean_entropy', 'mean_log_rank',
                    'rank_1', 'rank_2_10', 'rank_11_100', 'rank_101_1000', 'rank_gt_1000', 'curvature']

def token_stats(texts, model, tok, max_tokens=256, batch_size=8, device='cuda'):
    """Один проход модели по каждому тексту. Возвращает список словарей с массивами по токенам."""
    import torch
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    out = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors='pt', truncation=True, max_length=max_tokens,
                  padding=True, add_special_tokens=False).to(device)
        with torch.no_grad():
            logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask).logits.float()
        logp = torch.log_softmax(logits[:, :-1], dim=-1)          # прогноз для токенов 2..T
        target = enc.input_ids[:, 1:]
        lp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)    # log P(w_i | w_<i)
        p = logp.exp()
        ent = -(p * logp).sum(-1)                                  # энтропия, формула (2.2)
        var = (p * logp.pow(2)).sum(-1) - ent.pow(2)               # дисперсия log p по распределению модели
        rank = (logp > lp.unsqueeze(-1)).sum(-1) + 1               # ранг фактического токена
        mask = enc.attention_mask[:, 1:].bool()
        for b in range(len(batch)):
            m = mask[b]
            out.append({'ids': enc.input_ids[b][enc.attention_mask[b].bool()].cpu().numpy(),
                        'lp': lp[b][m].cpu().numpy(), 'ent': ent[b][m].cpu().numpy(),
                        'var': var[b][m].cpu().numpy(), 'rank': rank[b][m].cpu().numpy()})
        del logits, logp, p
    return out

def aggregate(s, k=None):
    """Признаки по первым k предсказанным токенам (k=None — все)."""
    lp, ent, var, rank = (s[x][:k] if k else s[x] for x in ('lp', 'ent', 'var', 'rank'))
    n = len(lp)
    if n == 0:
        return {f: 0.0 for f in LM_FEATURE_NAMES}
    curv = (lp.sum() + ent.sum()) / np.sqrt(max(var.sum(), 1e-8))   # (∑log p − ∑E[log p]) / √∑Var, E[log p] = −H
    return {
        'mean_lp': float(lp.mean()), 'std_lp': float(lp.std()), 'mean_entropy': float(ent.mean()),
        'mean_log_rank': float(np.log(rank).mean()),
        'rank_1': float((rank == 1).mean()), 'rank_2_10': float(((rank >= 2) & (rank <= 10)).mean()),
        'rank_11_100': float(((rank >= 11) & (rank <= 100)).mean()),
        'rank_101_1000': float(((rank >= 101) & (rank <= 1000)).mean()), 'rank_gt_1000': float((rank > 1000).mean()),
        'curvature': float(curv),
    }

In [ ]:
%%writefile /content/code/hybrid.py
"""Обучение и оценка статистического, стилометрического и гибридного детекторов (разделы 2.3, 3.3)."""
import json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from stylometry import stylometric_features, normalize, FEATURE_NAMES as STYLO
from lm_features import aggregate, LM_FEATURE_NAMES as LM
from metrics_utils import evaluate

VARIANTS = {'stylometric': STYLO, 'statistical': LM, 'hybrid': STYLO + LM}

def feature_table(texts, langs, stats, k=None, decode=None):
    """Признаки обеих групп. Если задан k, стилометрия считается по тексту первых k+1 токенов."""
    rows = []
    for t, l, s in zip(texts, langs, stats):
        if k is not None and decode is not None:
            t = decode(s['ids'][:k + 1], l)
        f = stylometric_features(t, l)
        f.update(aggregate(s, k))
        rows.append(f)
    return pd.DataFrame(rows)

def fit(X, y, sample_weight=None):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
    if sample_weight is None:
        return m.fit(X, y)
    return m.fit(X, y, logisticregression__sample_weight=sample_weight)

def threshold_at_fpr(model, X_val, y_val, fpr):
    """Порог, при котором на валидационной выборке доля ложных срабатываний равна fpr."""
    scores = model.predict_proba(X_val)[:, 1][np.asarray(y_val) == 0]
    return float(np.quantile(scores, 1 - fpr))

def train_eval_all(F_tr, y_tr, F_te, y_te, variants=VARIANTS):
    out, models = {}, {}
    for name, cols in variants.items():
        m = fit(F_tr[cols].values, y_tr)
        out[name] = evaluate(y_te, m.predict_proba(F_te[cols].values)[:, 1])
        models[name] = m
    return out, models

def export(models, thresholds, path, meta):
    """Веса гибридного детектора для расширения."""
    res = {'version': 'hybrid-v1', 'demo': False, 'meta': meta, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        res['langs'][lang] = {'features': STYLO + LM, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0]),
                              'thresholds': thresholds[lang]}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=1)

In [ ]:
%%writefile /content/code/transfer_hybrid.py
"""Межъязыковой перенос детекторов на признаках (серия Э2): стилометрический, статистический и гибридный.
Английский и русский детекторы используют разные языковые модели (GPT-2 и ruGPT-3), но одинаковый набор признаков,
поэтому классификатор, обученный на английских признаках, можно применить к русским."""
import numpy as np
from hybrid import fit, VARIANTS
from metrics_utils import evaluate


def moments(F):
    """Среднее и стандартное отклонение каждого признака (метки не нужны)."""
    mu = F.mean(0); sd = F.std(0); sd[sd == 0] = 1.0
    return mu, sd


def align(F_target, src_moments, tgt_moments):
    """Нормировка по языку: распределение каждого признака целевого языка приводится к среднему и разбросу
    исходного языка. Используются только неразмеченные тексты целевого языка."""
    (mu_s, sd_s), (mu_t, sd_t) = src_moments, tgt_moments
    return (F_target - mu_t) / sd_t * sd_s + mu_s


def zero_shot(F_en_tr, y_en_tr, F_ru_tr, F_ru_te, y_ru_te, variants=VARIANTS, n_boot=200):
    """Обучение только на английских текстах, проверка на русских: без адаптации и с нормировкой по языку."""
    out, models = {}, {}
    for name, cols in variants.items():
        m = fit(F_en_tr[cols].values, y_en_tr); models[name] = m
        Xte = F_ru_te[cols].values
        out[name] = {'без адаптации': evaluate(y_ru_te, m.predict_proba(Xte)[:, 1], n_boot=n_boot)}
        Xal = align(Xte, moments(F_en_tr[cols].values), moments(F_ru_tr[cols].values))
        out[name]['нормировка по языку'] = evaluate(y_ru_te, m.predict_proba(Xal)[:, 1], n_boot=n_boot)
    return out, models


def few_shot(F_en_tr, y_en_tr, F_ru_tr, y_ru_tr, F_ru_te, y_ru_te, idx, variants=VARIANTS):
    """idx — индексы n русских обучающих примеров. Варианты:
    'только русские' — классификатор обучается на n русских примерах;
    'англ. + русские' — на всех английских и n русских примерах, русские признаки нормированы по языку
    (по всем неразмеченным русским обучающим текстам), суммарный вес русских примеров равен весу английских."""
    out = {}
    y_ru_tr = np.asarray(y_ru_tr); y_en_tr = np.asarray(y_en_tr)
    for name, cols in variants.items():
        Xr = F_ru_tr[cols].values; Xe = F_en_tr[cols].values; Xte = F_ru_te[cols].values
        r = {}
        m = fit(Xr[idx], y_ru_tr[idx])
        r['только русские'] = evaluate(y_ru_te, m.predict_proba(Xte)[:, 1], n_boot=0)
        mom_en, mom_ru = moments(Xe), moments(Xr)
        X = np.vstack([Xe, align(Xr[idx], mom_en, mom_ru)])
        y = np.concatenate([y_en_tr, y_ru_tr[idx]])
        w = np.concatenate([np.ones(len(Xe)), np.full(len(idx), len(Xe) / len(idx))])
        m2 = fit(X, y, sample_weight=w)
        r['англ. + русские'] = evaluate(y_ru_te, m2.predict_proba(align(Xte, mom_en, mom_ru))[:, 1], n_boot=0)
        out[name] = r
    return out

In [ ]:
%%writefile /content/code/encoder.py
"""Дообучение многоязычного энкодера (XLM-RoBERTa) для обнаружения сгенерированного текста (серия Э2).
Обычный цикл обучения PyTorch: так меньше зависимости от версий библиотек, чем с Trainer."""
import math, time, random
import numpy as np


def set_seed(seed):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def encode(tok, texts, max_len=256):
    """Токенизация один раз заранее: список списков идентификаторов токенов (без дополнения)."""
    return tok(list(texts), truncation=True, max_length=max_len, add_special_tokens=True)['input_ids']


def _batch(ids_list, pad_id, device):
    import torch
    L = max(len(x) for x in ids_list)
    ids = torch.full((len(ids_list), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(ids_list), L), dtype=torch.long)
    for i, x in enumerate(ids_list):
        ids[i, :len(x)] = torch.tensor(x, dtype=torch.long); mask[i, :len(x)] = 1
    return ids.to(device), mask.to(device)


def train(init, ids, labels, pad_id, steps=None, epochs=None, batch_size=16, lr=2e-5, warmup=0.1,
          weight_decay=0.01, seed=0, device='cuda', log_every=100):
    """Дообучает классификатор. init — имя модели на Hugging Face или путь к сохранённой модели.
    Длительность задаётся либо числом шагов (steps), либо числом эпох (epochs)."""
    import torch
    from transformers import AutoModelForSequenceClassification
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(init, num_labels=2).to(device)
    labels = np.asarray(labels, dtype=np.int64); n = len(labels)
    per_epoch = math.ceil(n / batch_size)
    total = steps if steps is not None else per_epoch * epochs
    no_decay = ('bias', 'LayerNorm.weight', 'layer_norm.weight')
    groups = [{'params': [p for nm, p in model.named_parameters() if not any(k in nm for k in no_decay)], 'weight_decay': weight_decay},
              {'params': [p for nm, p in model.named_parameters() if any(k in nm for k in no_decay)], 'weight_decay': 0.0}]
    opt = torch.optim.AdamW(groups, lr=lr)
    n_warm = max(1, int(total * warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / n_warm if s < n_warm else max(0.0, (total - s) / max(1, total - n_warm)))
    use_amp = device == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    rng = np.random.default_rng(seed)
    model.train(); step = 0; t0 = time.time(); run_loss = []
    while step < total:
        order = rng.permutation(n)                      # новая перестановка на каждую эпоху
        for b in range(0, n, batch_size):
            if step >= total: break
            idx = order[b:b + batch_size]
            x, m = _batch([ids[i] for i in idx], pad_id, device)
            y = torch.tensor(labels[idx], device=device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=x, attention_mask=m).logits
            loss = torch.nn.functional.cross_entropy(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss.append(loss.item()); step += 1
            if log_every and step % log_every == 0:
                print(f'    шаг {step}/{total}  loss={np.mean(run_loss[-log_every:]):.3f}  {time.time() - t0:.0f} с')
    model.eval()
    return model


def predict(model, ids, pad_id, batch_size=64, device='cuda'):
    """Вероятность класса «сгенерирован» (метка 1) для каждого текста."""
    import torch
    out = []
    order = np.argsort([len(x) for x in ids])          # близкие по длине тексты в одном пакете — быстрее
    with torch.no_grad():
        for b in range(0, len(ids), batch_size):
            idx = order[b:b + batch_size]
            x, m = _batch([ids[i] for i in idx], pad_id, device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
                logits = model(input_ids=x, attention_mask=m).logits.float()
            out.append((idx, torch.softmax(logits, -1)[:, 1].cpu().numpy()))
    p = np.zeros(len(ids), dtype=np.float64)
    for idx, pr in out: p[idx] = pr
    return p


def free():
    """Освобождает память видеокарты. Перед вызовом удалите модель: del model."""
    import torch, gc
    gc.collect(); torch.cuda.empty_cache()


def few_shot_sample(labels, n, seed):
    """Индексы n примеров (по n/2 каждого класса) из обучающей выборки."""
    rng = np.random.default_rng(seed); labels = np.asarray(labels)
    pos = rng.choice(np.where(labels == 1)[0], n // 2, replace=False)
    neg = rng.choice(np.where(labels == 0)[0], n - n // 2, replace=False)
    return rng.permutation(np.concatenate([pos, neg]))

## 3. Данные и признаки (те же выборки, что в шагах 2–3)

In [ ]:
import importlib, pickle, time, json, numpy as np, pandas as pd
import stylometry, metrics_utils, baseline_stylometry, lm_features, hybrid, transfer_hybrid, encoder
for m in (stylometry, metrics_utils, baseline_stylometry, lm_features, hybrid, transfer_hybrid, encoder): importlib.reload(m)
from stylometry import normalize
from baseline_stylometry import split
from hybrid import feature_table, VARIANTS
from metrics_utils import evaluate
from transfer_hybrid import zero_shot, few_shot
from encoder import encode, train, predict, free, few_shot_sample
SEED = 42
df = pd.read_parquet(f'{WORK}/dataset_step1.parquet'); df = df[df.dataset == 'M4']
def bal(d, cap): return pd.concat([g.sample(min(len(g), cap), random_state=SEED) for _, g in d.groupby('label')])
data = {'en': bal(df[df.lang == 'en'], 6000), 'ru': df[df.lang == 'ru']}
for k, d in data.items():
    d = d.copy(); d['norm'] = d.text.astype(str).map(normalize); data[k] = d.reset_index(drop=True)
STATS = pickle.load(open(f'{WORK}/lm_stats_step2.pkl', 'rb'))
assert len(STATS['en']) == len(data['en']) and len(STATS['ru']) == len(data['ru']), 'Кэш не совпадает с данными — напиши мне'
SPL, FEAT, Y = {}, {}, {}
for lang in ['en', 'ru']:
    d = data[lang]; tr, va, te = split(d, SEED)
    SPL[lang] = {'tr': tr.index.values, 'va': va.index.values, 'te': te.index.values}
    FEAT[lang] = feature_table(d.norm, [lang] * len(d), STATS[lang]); Y[lang] = d.label.values.astype(int)
    print(lang, {k: (len(v), int(Y[lang][v].sum())) for k, v in SPL[lang].items()}, '(всего, из них ИИ)')
assert len(SPL['ru']['te']) == 1350 and len(SPL['en']['te']) == 1803, 'Разбиение не совпало с шагами 2–3 — напиши мне'
NS = [100, 500, 1000]; SEEDS = [0, 1, 2]
RES = {'hybrid_features': {}, 'encoder': {}}
def save():
    json.dump(RES, open(f'{WORK}/results_step4.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2, default=str)
def short(r): return f"AUROC={r['AUROC']:.3f} F1={r['F1']:.3f} FPR={r['FPR']:.3f} TPR@1%={r['TPR@1%FPR']:.3f}"

## 4. Детекторы на признаках: перенос английский → русский
«Нормировка по языку» — каждый признак русских текстов приводится к среднему и разбросу английских. Для этого нужны только **неразмеченные** русские тексты (метки не используются), поэтому это честный вариант «без русских примеров».

In [ ]:
en, ru = SPL['en'], SPL['ru']
Fe, ye = FEAT['en'].iloc[en['tr']], Y['en'][en['tr']]
Fr, yr = FEAT['ru'].iloc[ru['tr']], Y['ru'][ru['tr']]
Ft, yt = FEAT['ru'].iloc[ru['te']], Y['ru'][ru['te']]
zs, _ = zero_shot(Fe, ye, Fr, Ft, yt)
RES['hybrid_features']['zero_shot'] = zs
print('Обучение только на английском, проверка на русском тесте:')
for name, v in zs.items():
    for mode, r in v.items(): print(f'  {name:12s} {mode:20s} {short(r)}')
fsr = {}
for n in NS + ['all']:
    fsr[str(n)] = []
    for s in (SEEDS if n != 'all' else [0]):
        idx = np.arange(len(yr)) if n == 'all' else few_shot_sample(yr, n, s)
        fsr[str(n)].append(few_shot(Fe, ye, Fr, yr, Ft, yt, idx))
RES['hybrid_features']['few_shot'] = fsr; save()
print('\nДообучение на русских примерах (AUROC, среднее ± разброс по 3 повторам):')
for name in VARIANTS:
    for mode in ['только русские', 'англ. + русские']:
        row = []
        for n in NS + ['all']:
            a = [r[name][mode]['AUROC'] for r in fsr[str(n)]]
            row.append(f'{n}: {np.mean(a):.3f}' + (f'±{np.std(a):.3f}' if len(a) > 1 else ''))
        print(f'  {name:12s} {mode:16s}', ' | '.join(row))

## 5. Энкодер XLM-RoBERTa: обучение на английском
2 эпохи на ≈8400 английских текстах (≈5–8 минут). Если обучение «не пошло» (так иногда бывает у XLM-RoBERTa: модель начинает всегда отвечать одно и то же), ячейка сама перезапускает его с другим начальным значением.

In [ ]:
from transformers import AutoTokenizer
ENC = 'FacebookAI/xlm-roberta-base'
tok = AutoTokenizer.from_pretrained(ENC); PAD = tok.pad_token_id
IDS = {lang: encode(tok, data[lang].norm, max_len=256) for lang in ['en', 'ru']}
print('Доля текстов длиннее 256 токенов:', {l: round(float(np.mean([len(x) >= 256 for x in IDS[l]])), 3) for l in IDS})
take = lambda lang, part: [IDS[lang][i] for i in SPL[lang][part]]
y_of = lambda lang, part: Y[lang][SPL[lang][part]]
t0 = time.time()
for attempt, s in enumerate([0, 1, 2]):
    model = train(ENC, take('en', 'tr'), y_of('en', 'tr'), PAD, epochs=2, seed=s, log_every=200)
    from sklearn.metrics import roc_auc_score
    val_auc = roc_auc_score(y_of('en', 'va'), predict(model, take('en', 'va'), PAD))
    print(f'  попытка {attempt + 1}: AUROC на английской валидации = {val_auc:.3f}')
    if val_auc >= 0.75 or attempt == 2: break
    del model; free()
print(f'Обучение на английском: {time.time() - t0:.0f} с')
model.save_pretrained('/content/enc_en'); tok.save_pretrained('/content/enc_en')
RES['encoder']['en_seed'] = s; RES['encoder']['en_val_auroc'] = float(val_auc)
RES['encoder']['en_test'] = evaluate(y_of('en', 'te'), predict(model, take('en', 'te'), PAD))
RES['encoder']['zero_shot_ru'] = evaluate(y_of('ru', 'te'), predict(model, take('ru', 'te'), PAD))
del model; free(); save()
print('Английский тест:              ', short(RES['encoder']['en_test']))
print('Русский тест без русских примеров:', short(RES['encoder']['zero_shot_ru']))

## 6. Энкодер: дообучение на русских примерах (≈25–35 минут)
Для каждого числа примеров (100, 500, 1000) и каждого из 3 повторов:
- **только русские** — исходный XLM-RoBERTa обучается на n русских примерах;
- **англ. → + русские** — энкодер из ячейки 5 доучивается на тех же n русских примерах.

Длительность обучения одинакова для всех n: 150 шагов по 16 примеров. В конце — обучение на всех ≈6300 русских обучающих текстах (2 эпохи) как верхняя граница.

In [ ]:
yr_all = y_of('ru', 'tr'); ids_tr = take('ru', 'tr'); ids_te = take('ru', 'te'); y_te = y_of('ru', 'te')
ids_en_te = take('en', 'te'); y_en_te = y_of('en', 'te')
RES['encoder'].setdefault('few_shot', {})
for n in NS + ['all']:
    for s in (SEEDS if n != 'all' else [0]):
        key = f'{n}_seed{s}'
        if key in RES['encoder']['few_shot']: continue          # уже сделано (повторный запуск ячейки)
        idx = np.arange(len(yr_all)) if n == 'all' else few_shot_sample(yr_all, n, s)
        ids_n = [ids_tr[i] for i in idx]; y_n = yr_all[idx]
        kw = dict(epochs=2) if n == 'all' else dict(steps=150)
        row = {}
        for mode, init in [('только русские', ENC), ('англ. → + русские', '/content/enc_en')]:
            t0 = time.time()
            m = train(init, ids_n, y_n, PAD, seed=s, log_every=0, **kw)
            row[mode] = evaluate(y_te, predict(m, ids_te, PAD), n_boot=200 if n == 'all' else 0)
            if mode == 'англ. → + русские':                       # не «забыл» ли энкодер английский
                row['англ. → + русские: английский тест'] = evaluate(y_en_te, predict(m, ids_en_te, PAD), n_boot=0)
            del m; free()
            print(f'n={n} повтор {s} | {mode:18s} {short(row[mode])}  ({time.time() - t0:.0f} с)')
        RES['encoder']['few_shot'][key] = row; save()

## 7. Сводка и рисунок

In [ ]:
import matplotlib.pyplot as plt
XS = [0] + NS + ['all']; XL = ['0'] + [str(n) for n in NS] + [f'все\n({len(SPL["ru"]["tr"])})']
def series(get):
    m, sd = [], []
    for n in XS:
        vals = get(n)
        m.append(np.mean(vals) if vals else np.nan); sd.append(np.std(vals) if len(vals) > 1 else 0)
    return np.array(m), np.array(sd)
enc = RES['encoder']['few_shot']; hf = RES['hybrid_features']
def enc_get(mode):
    def g(n):
        if n == 0: return [RES['encoder']['zero_shot_ru']['AUROC']] if mode.startswith('англ') else []
        return [v[mode]['AUROC'] for k, v in enc.items() if k.split('_')[0] == str(n)]
    return g
def hyb_get(mode):
    def g(n):
        if n == 0: return [hf['zero_shot']['hybrid']['нормировка по языку']['AUROC']] if mode.startswith('англ') else []
        return [r['hybrid'][mode]['AUROC'] for r in hf['few_shot'][str(n)]]
    return g
lines = {'Энкодер: только русские': enc_get('только русские'),
         'Энкодер: англ. → + русские': enc_get('англ. → + русские'),
         'Гибрид: только русские': hyb_get('только русские'),
         'Гибрид: англ. + русские': hyb_get('англ. + русские')}
summary = {}
fig, ax = plt.subplots(figsize=(8, 4.5)); x = np.arange(len(XS))
for (name, get), style in zip(lines.items(), ['o-', 's-', 'o--', 's--']):
    m, sd = series(get); summary[name] = {str(n): (None if np.isnan(a) else round(float(a), 4)) for n, a in zip(XS, m)}
    ax.errorbar(x, m, yerr=sd, fmt=style, capsize=3, label=name)
ax.axhline(0.5, color='grey', lw=0.8, ls=':'); ax.set_xticks(x); ax.set_xticklabels(XL)
ax.set_xlabel('число русских обучающих примеров'); ax.set_ylabel('AUROC на русском тесте'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(f'{WORK}/e2_curve.png', dpi=200); plt.show()
RES['summary_auroc'] = summary; save()
print(pd.DataFrame(summary).T.to_string())
fg = [v['англ. → + русские: английский тест']['AUROC'] for k, v in enc.items() if 'англ. → + русские: английский тест' in v]
print('\nЭнкодер на английском тесте: до дообучения', round(RES['encoder']['en_test']['AUROC'], 3),
      '| после дообучения на русских (мин–макс):', round(min(fg), 3), '–', round(max(fg), 3))
print('Сохранено: results_step4.json, e2_curve.png')